# 🔍 Engenharia Reversa da Busca de Emprego
**Projeto:** Análise do Mercado de Dados Brasileiro  
**Autores:** Jonatas Oliveira de Lima & Guilherme Soares Santos  
**Curso:** Ciência de Dados — FATEC Santana de Parnaíba  
**Fonte dos dados:** Adzuna.com.br  

---

## Etapas deste notebook
1. Carregamento dos dados
2. Diagnóstico inicial
3. Limpeza dos dados
4. Categorização dos títulos
5. Análise exploratória com gráficos

---
## 1. Importações e Carregamento

In [ ]:
# Bibliotecas de manipulação de dados
import pandas as pd
import numpy as np

# Bibliotecas de visualização
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configurações visuais
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# Carrega o CSV coletado pelo scraper
df = pd.read_csv('vagas_completas.csv', low_memory=False)

print(f'✅ Dados carregados: {df.shape[0]} linhas × {df.shape[1]} colunas')
df.head()

---
## 2. Diagnóstico Inicial

Antes de limpar, precisamos **entender o estado atual dos dados**:  
quantos nulos existem, quantas duplicatas, quais os tipos de cada coluna.

In [ ]:
# Visão geral: tipos de dados e valores não-nulos
print('=' * 50)
print('  INFORMAÇÕES GERAIS DO DATAFRAME')
print('=' * 50)
df.info()

In [ ]:
# Contagem de valores nulos por coluna
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(1)

diagnostico = pd.DataFrame({
    'Nulos': nulos,
    '% de nulos': pct_nulos
})

print('\n📊 Nulos por coluna:')
display(diagnostico)

# Duplicatas
total_duplicatas = df.duplicated().sum()
print(f'\n🔁 Duplicatas encontradas: {total_duplicatas} ({total_duplicatas/len(df)*100:.1f}%)')

---
## 3. Limpeza dos Dados

Com o diagnóstico em mãos, aplicamos as correções:
- Remover duplicatas
- Padronizar o título (maiúsculas/minúsculas, espaços extras)
- Tratar colunas com muitos nulos
- Resetar o índice

In [ ]:
total_antes = len(df)

# ── 3.1 Remover duplicatas ──────────────────────────────────
# Considera duplicata quando Titulo E Link são iguais
df = df.drop_duplicates(subset=['Titulo', 'Link'])
print(f'🗑️  Duplicatas removidas : {total_antes - len(df)}')
print(f'   Registros restantes  : {len(df)}')

# ── 3.2 Padronizar o Título ─────────────────────────────────
# strip()  → remove espaços no início/fim
# title()  → capitaliza cada palavra (ex: "ANALISTA DE DADOS" → "Analista De Dados")
df['Titulo'] = df['Titulo'].str.strip().str.title()
print('\n✅ Títulos padronizados (strip + title case)')

# ── 3.3 Colunas com nulos ───────────────────────────────────
# Data_Publicacao e Requisitos têm >99% de nulos → descartamos
colunas_descartar = ['Data_Publicacao', 'Requisitos', 'Status']
df = df.drop(columns=colunas_descartar)
print(f'\n🗑️  Colunas descartadas (>99% nulos ou sem variação): {colunas_descartar}')

# ── 3.4 Salário: marcar se informado ou não ─────────────────
df['Salario_Informado'] = df['Salario_Texto'].apply(
    lambda x: 'Não informado' if str(x).strip().lower() == 'não informado' else 'Informado'
)

# ── 3.5 Resetar índice ──────────────────────────────────────
df = df.reset_index(drop=True)

print(f'\n✅ Limpeza concluída! Dataset final: {df.shape[0]} vagas × {df.shape[1]} colunas')
df.head()

---
## 4. Categorização dos Títulos

Usamos **Regex** (expressões regulares) para identificar padrões nos títulos  
e classificar cada vaga em uma categoria de nível de carreira.

In [ ]:
def categorizar_titulo(titulo):
    """
    Classifica o título da vaga em uma categoria de carreira.
    A ordem importa: verifica do mais específico para o mais genérico.
    """
    t = str(titulo).lower()

    # Bolsas de pesquisa (graduando, graduado, mestre, doutor)
    if any(p in t for p in ['bolsista', 'bolsa', 'graduando', 'graduado']):
        return 'Bolsa de Pesquisa'

    # Estágios e trainees
    if any(p in t for p in ['estágio', 'estagio', 'trainee', 'aprendiz']):
        return 'Estágio / Trainee'

    # Nível sênior
    if any(p in t for p in ['sênior', 'senior', 'sr.', ' sr ']):
        return 'Sênior'

    # Nível pleno
    if any(p in t for p in ['pleno', 'pl.', ' pl ']):
        return 'Pleno'

    # Nível júnior
    if any(p in t for p in ['júnior', 'junior', 'jr.', ' jr ']):
        return 'Júnior'

    # Cargos de liderança
    if any(p in t for p in ['gerente', 'coordenador', 'diretor', 'head', 'líder', 'lider']):
        return 'Liderança'

    # Cargos técnicos sem nível explícito
    if any(p in t for p in ['analista', 'cientista', 'engenheiro', 'especialista', 'arquiteto']):
        return 'Técnico (sem nível)'

    return 'Outros'


# Aplica a função em cada linha da coluna Titulo
df['Categoria'] = df['Titulo'].apply(categorizar_titulo)

print('✅ Categorização concluída!')
print('\n📊 Distribuição por categoria:')
print(df['Categoria'].value_counts())

---
## 5. Análise Exploratória

Com os dados limpos e categorizados, geramos os gráficos principais.

In [ ]:
# ── Gráfico 1: Distribuição por Categoria de Vaga ───────────
contagem = df['Categoria'].value_counts()
pct = (contagem / contagem.sum() * 100).round(1)

fig, ax = plt.subplots()
bars = ax.barh(contagem.index, contagem.values, color=sns.color_palette('muted', len(contagem)))

# Adiciona o valor e % em cada barra
for bar, valor, p in zip(bars, contagem.values, pct.values):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{valor:,} ({p}%)', va='center', fontsize=10)

ax.set_title('Distribuição de Vagas por Categoria de Carreira', fontsize=14, fontweight='bold')
ax.set_xlabel('Quantidade de Vagas')
ax.set_xlim(0, contagem.max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 2: Transparência Salarial ───────────────────────
sal = df['Salario_Informado'].value_counts()
cores = ['#2ecc71', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza
axes[0].pie(
    sal.values,
    labels=sal.index,
    autopct='%1.1f%%',
    colors=cores,
    startangle=90,
    textprops={'fontsize': 12}
)
axes[0].set_title('Transparência Salarial', fontsize=13, fontweight='bold')

# Barras
axes[1].bar(sal.index, sal.values, color=cores)
for i, (idx, val) in enumerate(sal.items()):
    axes[1].text(i, val + 20, str(val), ha='center', fontsize=11)
axes[1].set_title('Vagas com/sem salário divulgado', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

nao_informado = sal.get('Não informado', 0)
pct_opaco = nao_informado / len(df) * 100
print(f'\n💡 Insight: {pct_opaco:.1f}% das vagas não divulgam o salário.')
print('   Isso confirma a opacidade estrutural do mercado de dados no Brasil.')

In [ ]:
# ── Gráfico 3: Salário Médio por Categoria ───────────────────
# Apenas vagas que têm salário informado
df_com_salario = df[df['Salario_Medio'].notna()].copy()

salario_categoria = (
    df_com_salario.groupby('Categoria')['Salario_Medio']
    .mean()
    .sort_values(ascending=False)
    .round(0)
)

fig, ax = plt.subplots()
bars = ax.bar(
    salario_categoria.index,
    salario_categoria.values,
    color=sns.color_palette('coolwarm', len(salario_categoria))
)

for bar, val in zip(bars, salario_categoria.values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 100,
        f'R$ {val:,.0f}',
        ha='center', fontsize=10, fontweight='bold'
    )

ax.set_title('Salário Médio por Categoria (vagas com salário divulgado)', fontsize=13, fontweight='bold')
ax.set_ylabel('Salário Médio (R$)')
ax.set_xlabel('Categoria')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 4: Top 10 Títulos Mais Frequentes ───────────────
top10 = df['Titulo'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top10.index[::-1], top10.values[::-1],
               color=sns.color_palette('Blues_d', 10))

for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)

ax.set_title('Top 10 Títulos de Vagas Mais Frequentes', fontsize=13, fontweight='bold')
ax.set_xlabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
# ── Resumo Final ─────────────────────────────────────────────
print('=' * 55)
print('  RESUMO DA ANÁLISE — MERCADO DE DADOS NO BRASIL')
print('=' * 55)
print(f'  Total de vagas analisadas   : {len(df):,}')
print(f'  Vagas sem salário divulgado : {pct_opaco:.1f}%')
print(f'  Categorias identificadas    : {df["Categoria"].nunique()}')
print()

print('  Top 3 categorias:')
for cat, qtd in contagem.head(3).items():
    print(f'    → {cat}: {qtd:,} vagas ({qtd/len(df)*100:.1f}%)')

print()
print('  💡 Principal insight:')
print('     A maior parte das vagas são bolsas de pesquisa,')
print('     revelando que o mercado formal CLT ainda é restrito')
print('     e a opacidade salarial domina o setor.')
print('=' * 55)

---
## 6. Exportar Dataset Limpo

In [ ]:
# Salva o dataset limpo e categorizado para uso no dashboard
df.to_csv('vagas_limpo.csv', index=False, encoding='utf-8-sig')
print(f'✅ Dataset limpo exportado: vagas_limpo.csv ({len(df):,} vagas)')